### This notebook creates the job that automates running the SQL and R notebooks
First we need to load (and install if not present) packages

In [0]:
%r
library(tidyverse)

install.packages("remotes")
library(remotes)

remotes::install_github("databrickslabs/databricks-sdk-r")
library(databricks)

Create a widget that stores your personal access token (PAT) that you have used to connect to databricks. If you have completed the previous exercises, you will have a PAT stored in your .Renviron file, otherwise details on how to create a new PAT can be found on the [Analysts Guide](https://dfe-analytical-services.github.io/analysts-guide/ADA/databricks_rstudio_sql_warehouse.html#databricks-token). 

**IMPORTANT NOTE: Do not include your personal access token in the code.** We use a text widget to specify our PAT to ensure we don't expose it. If it were coded in the notebook then anyone with access to the notebook would be able to use it to emulate our Databricks access permissions. Databricks has in-built protections for this, so it should throw an error if you try to push a commit that contains the token (either in the code or saved as a widget), so any references in the code or the widget value box must be removed before committing to a remote repository.

In [0]:
%py
dbutils.widgets.text("api_token", "")

### Connect to databricks API
Connect to the Databricks API using the databricks package and get cluster details needed to run the job. The cluster_id variable must only contain one value - here we're automating this step by filtering to our Personal Compute Cluster. Remember that we need to use a Personal Compute Cluster with Dedicated (Single User) access to use to run R cells. 

In [0]:
%r
host <- "https://adb-5037484389568426.6.azuredatabricks.net/" 

api_token <- dbutils.widgets.get("api_token")  

client <- databricks::DatabricksClient(host = host, token = api_token)  

clusters <- databricks::clustersList(client)  

# Get cluster variables
clusters <- clusters %>% 
       select(cluster_id, 
              creator_user_name, 
              cluster_name, 
              spark_version, 
              cluster_cores, 
              cluster_memory_mb) 

# Filter to your Personal Compute Cluster 
# (in this case I named my R-Studio cluster so that this step could be automated, filtering)
personal_compute_cluster <- clusters %>%
       filter(str_detect(cluster_name, "RStudio"))
        
# Get your Personal Compute Cluster ID
cluster_id <- personal_compute_cluster %>%
       select(cluster_id) %>%
       toString()

# Check that there is only one output, and confirm that it is the right cluster
print(personal_compute_cluster)

### Define parameters of the job that we want to run
Define the job name, as well as the path to the notebooks that we need to run, and a task key for each job.

In [1]:
%r
job_name <- "Average pupil premium pipeline" 

# NOTE: you will need to update the filepath to the correct location in your Databricks workspace
sql_notebook_path <- "/Workspace/Users/robert.tarrant@education.gov.uk/techskills_workshops/Databricks_workshops/Databricks workshop/03 Migrating existing processes/hybrid_pipeline_solutions/1_databricks_job_solution/source_code/1_sql_code.ipynb" 
r_notebook_path <- "/Workspace/Users/robert.tarrant@education.gov.uk/techskills_workshops/Databricks_workshops/Databricks workshop/03 Migrating existing processes/hybrid_pipeline_solutions/1_databricks_job_solution/source_code/2_r_code.ipynb" 

task_key_sql <- "sql_key" 
task_key_r <- "r_key"

: [1m[33mError[39m:[22m
[33m![39m Syntax error: unexpected input

### Define the tasks
We can then define the tasks as lists containing the task key, cluster ID, and the type of task (in this case a notebook task) including it's details. 

As we only want the R notebook to run if the SQL notebook has completed successfully, we have added the depends_on and run_if arguments to the list for the R job task.

In [0]:
%r
# First the SQL job
sql_job_task <- list(
 task_key = task_key_sql,
 existing_cluster_id = cluster_id,
 notebook_task = list(
                  notebook_path = sql_notebook_path,
                  source = "WORKSPACE"
                  )
 )
 
# Then the R job
r_job_task <- list(
 task_key = task_key_r,
 existing_cluster_id = cluster_id,
 notebook_task = list(
                  notebook_path = r_notebook_path,
                  source = "WORKSPACE"
                  ),
  depends_on = list(task_key = task_key_sql),
  run_if = "ALL_SUCCESS"
 )

### Create the job
After defining the tasks, we can create the job, passing the client, the job name, and the tasks created above. We can also add an argument to send an emaail update when the job has successfully run (though you will need to make sure the email is actually yours!). 

The function returns the ID of the job we just created, so we will want to store the response in a variable called workflow so we can refer to it later.

In [0]:
%r
workflow <- jobsCreate(client,
                       name = job_name,
                       tasks = list(sql_job_task, #list
                                    r_job_task), #list
                       email_notifications = list(
                         on_success = c("your.name@education.gov.uk") # <-- please make sure to update to your email!
                         )
                       )

### Execute the job
To start running the job we will need the job_id, which we can now extract from the workflow object that we have saved. We can run the job using the databricks::jobsRunNow function, making sure to assign the output to an object to store the run_id associated with the job.

In [0]:
%r
job_id <- workflow$job_id  

job_run <- jobsRunNow(client,
                      job_id = job_id)

job_run_id <- job_run$run_id

### Check on the progress of the job
We can check on the progress of the job by defining a job link, and a job run link, as detailed below. Click on the links below to check on the progress of the job. 

In [0]:
%r
# Show the job link to view the job (right click to open in a new tab)
job_link <- paste0(host,"jobs/",job_id)
job_link

In [0]:
%r
# Show the job link to view the job (right click to open in a new tab)
job_run_link <- paste0(job_link,"/runs/", job_run_id)
job_run_link